In [ ]:
import asyncio
import datetime
import json
import subprocess
import threading
import statistics
import uuid

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Self, override

import lsst.rsp

# Parameters and globals
URL='https://sdfdatas3.slac.stanford.edu/rubin-dp02-products/u/ctslater/DM-33345-tract4431/20220125T013610Z/matchedCatalogTract/4431/u/matchedCatalogTract_LSSTCam-imSim_4431_u_DC2_u_ctslater_DM-33345-tract4431_20220125T013610Z.fits?AWSAccessKeyId=dp02user&Signature=dCOzz%2BmwVzAERvPM74asZWniIOM%3D&Expires=1774406495'
FILESIZE=1334496960
CURL_CONCURRENCY=100
CMD="curl"
OUTPUT_DIR="/rubin/shared/speedtest"

lock=threading.Lock()


@dataclass
class Results:
    mean: float
    median: float
    aggregate: float
    effective: float
    real: float | None = None
    node: str | None = None
    start: float | None = None
    concurrency: int | None = None
    end: float | None = None
    
    @classmethod
    def from_list(cls, inp: list[float]) -> Self:
        """The list will contain the stuff curl can tell us.
        The other fields have to be filled in from the
        caller."""
        mean = statistics.mean(inp)
        median = statistics.median(inp)
        aggregate = sum(inp)
        effective = len(inp) * min(inp)
        return cls(
            mean=mean,
            median=median,
            aggregate=aggregate,
            effective=effective,
            real=None,
            node=None,
            concurrency=None,
            start=None,
            end=None,
        )

    @override
    def __str__(self) -> str:
        ret = (f"mean: {self.mean:02.2f}MBps"
               f" median: {self.median:02.2f}MBps"
               f" aggregate {self.aggregate:02.2f}MBps"
               f" effective {self.effective:02.2f}MBps"
              )
        if self.real:
            ret += f" real {self.real:02.2f}MBps"
        if self.node:
            ret += f" node {self.node}"
        if self.concurrency:
            ret += f" concurrency {self.concurrency}"
        if self.start:
            ret += f" start {datetime.datetime.fromtimestamp(self.start, tz=datetime.UTC)}"
        if self.end:
            ret += f" end {datetime.datetime.fromtimestamp(self.end, tz=datetime.UTC)}"
        return ret
        
multiprocess_results: dict[int, Results] = {}
results: list[float] = []

async def run_curl(cmd_args: list[str], results: list[float]) -> None:
    proc = await asyncio.create_subprocess_exec(CMD, *cmd_args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    await proc.wait()
    out = (await proc.stdout.read()).decode()
    err = (await proc.stderr.read()).decode()
    speed = err.split("\r")[-1].rstrip().split()[-1]
    if speed.endswith("k"):
        # convert it
        print(f"Converting speed in {speed} k to M")
        f_speed=(float((err.split("\r")[-1].rstrip().split()[-1])[:-1]))
        f_speed = f_speed / 2**10  # Convert to MBbps
        lock.acquire()
        results.append(f_speed)
        lock.release()
    elif speed.endswith("M"):
        # Last line of results is completed download.  Last field is download speed.
        # Last character of that is "M", for MBps, or "k", for Kbps
        f_speed=float((err.split("\r")[-1].rstrip().split()[-1])[:-1])
        print(f"speed: {f_speed}MBps")
        lock.acquire()
        results.append(f_speed)
        lock.release()
    else:
        print(f"Uninterpretable speed {speed}")
    return
    
async def spawn_curls(concurrency: int, cmd_args: list[str]) -> Results:
    results: list[float] = []
    async with asyncio.TaskGroup() as tg:
        tasks: set[asyncio.Task] = set()
        then=datetime.datetime.now(tz=datetime.UTC)
        for task in range(concurrency):
            tasks.add(tg.create_task(run_curl(cmd_args, results)))

    now=datetime.datetime.now(tz=datetime.UTC)
    node=lsst.rsp.get_node()
    res=Results.from_list(results)
    res.node=node
    realspeed=((concurrency*FILESIZE) / (now - then).total_seconds()) / 2**20
    res.real = realspeed
    res.start = then.timestamp()
    res.end = now.timestamp()
    res.concurrency=concurrency
    return res


async def run_test(concurrency: int) -> None:
    cmd_args=[ "--output", "/dev/null", "--parallel", "--parallel-max", str(CURL_CONCURRENCY), "--parallel-immediate", "--url", URL ]
    results = []  # Reset for each run
    pl="" if concurrency < 2 else "s in parallel"
    print(f"{concurrency} curl{pl}")
    then=datetime.datetime.now(tz=datetime.UTC)
    p_results = await spawn_curls(concurrency, cmd_args)   
    lock.acquire()
    multiprocess_results[concurrency] = p_results
    lock.release()


async def multi_test() -> None:
    for concurrency in [ 1, 3, 10, 30]:
        # 100 doesn't complete on a 4CPU 16GB Pod
        await run_test(concurrency)

In [ ]:
# Run the test
await multi_test()

# Write output
o_dir = Path(OUTPUT_DIR)
if not o_dir.is_dir():
    o_dir.mkdir()
    o_dir.chmod(0o777)  # Yeah, world-writeable.
for k,v in multiprocess_results.items():
    f_uuid=uuid.uuid4()
    # Write to a random filename; we're gonna suck it all up with glob() anyway
    fname = f"{f_uuid}-results.json"
    o_file = o_dir / fname
    o_file.write_text(json.dumps(asdict(v)))
    print(v)